In [15]:
import torch

print("PyTorch imported.")

PyTorch imported.


In [16]:
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

print("Libraries imported.")

Libraries imported.


In [17]:
dataset_path = Path(
    r"C:\Users\saman\OneDrive\Desktop\OG flower dataset\archive\102 flower"
)

flowers_path = dataset_path / "flowers"

print("Dataset path:")
print(dataset_path)

print("\nDataset exists:")
print(dataset_path.exists())

Dataset path:
C:\Users\saman\OneDrive\Desktop\OG flower dataset\archive\102 flower

Dataset exists:
True


In [18]:
from PIL import Image, ImageOps

class ResizeAndPad:
    
    def __init__(self, size):
        self.size = size

    def __call__(self, image):
        
        width, height = image.size
        
        scale = min(self.size / width, self.size / height)
        
        new_width = int(width * scale)
        new_height = int(height * scale)
        
        image = image.resize(
            (new_width, new_height),
            Image.Resampling.LANCZOS
        )
        
        left = (self.size - new_width) // 2
        right = self.size - new_width - left
        top = (self.size - new_height) // 2
        bottom = self.size - new_height - top
        
        image = ImageOps.expand(
            image,
            border=(left, top, right, bottom),
            fill="black"
        )
        
        return image


resize_and_pad = ResizeAndPad(224)

print("Resize + Padding transformation created.")

Resize + Padding transformation created.


In [19]:
train_transforms = transforms.Compose([
    transforms.Lambda(resize_and_pad),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

valid_test_transforms = transforms.Compose([
    transforms.Lambda(resize_and_pad),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Preprocessing pipelines created.")

Preprocessing pipelines created.


In [20]:
train_dataset = datasets.ImageFolder(
    flowers_path / "train",
    transform=train_transforms
)

valid_dataset = datasets.ImageFolder(
    flowers_path / "valid",
    transform=valid_test_transforms
)

test_dataset = datasets.ImageFolder(
    flowers_path / "test",
    transform=valid_test_transforms
)

print("Training images:", len(train_dataset))
print("Validation images:", len(valid_dataset))
print("Test images:", len(test_dataset))
print("Number of classes:", len(train_dataset.classes))

Training images: 6552
Validation images: 818
Test images: 819
Number of classes: 102


In [21]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(valid_loader))
print("Test batches:", len(test_loader))

Training batches: 205
Validation batches: 26
Test batches: 26


In [22]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cpu


In [23]:
from torchvision import models

resnet18 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

resnet50 = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

print("ResNet-18 loaded.")
print("ResNet-50 loaded.")


ResNet-18 loaded.
ResNet-50 loaded.


In [24]:
resnet18.fc = torch.nn.Linear(
    resnet18.fc.in_features,
    102
)

resnet50.fc = torch.nn.Linear(
    resnet50.fc.in_features,
    102
)

print("ResNet-18 final layer: 102 classes")
print("ResNet-50 final layer: 102 classes")

ResNet-18 final layer: 102 classes
ResNet-50 final layer: 102 classes


In [25]:
for parameter in resnet18.parameters():
    parameter.requires_grad = False

for parameter in resnet50.parameters():
    parameter.requires_grad = False

# Allow the new final layers to be trained
for parameter in resnet18.fc.parameters():
    parameter.requires_grad = True

for parameter in resnet50.fc.parameters():
    parameter.requires_grad = True

print("Pretrained layers frozen.")
print("Final classification layers trainable.")

Pretrained layers frozen.
Final classification layers trainable.


In [26]:
criterion = torch.nn.CrossEntropyLoss()

print("Cross-entropy loss created.")

Cross-entropy loss created.


In [27]:
optimizer_resnet18 = torch.optim.SGD(
    resnet18.fc.parameters(),
    lr=0.001
)

print("ResNet-18 SGD optimizer created.")

ResNet-18 SGD optimizer created.


In [28]:
optimizer_resnet50 = torch.optim.SGD(
    resnet50.fc.parameters(),
    lr=0.001
)

print("ResNet-50 SGD optimizer created.")

ResNet-50 SGD optimizer created.


In [29]:
resnet18_trainable = sum(
    parameter.numel()
    for parameter in resnet18.parameters()
    if parameter.requires_grad
)

resnet50_trainable = sum(
    parameter.numel()
    for parameter in resnet50.parameters()
    if parameter.requires_grad
)

print("ResNet-18 trainable parameters:", resnet18_trainable)
print("ResNet-50 trainable parameters:", resnet50_trainable)

ResNet-18 trainable parameters: 52326
ResNet-50 trainable parameters: 208998


In [30]:
train_dataset = datasets.ImageFolder(
    flowers_path / "train",
    transform=train_transforms
)

print("Training dataset recreated.")
print("Training images:", len(train_dataset))

Training dataset recreated.
Training images: 6552


In [31]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

print("Training DataLoader recreated.")
print("Training batches:", len(train_loader))

Training DataLoader recreated.
Training batches: 205


In [32]:
images, labels = next(iter(train_loader))

outputs18 = resnet18(images)
outputs50 = resnet50(images)

print("ResNet-18 output shape:", outputs18.shape)
print("ResNet-50 output shape:", outputs50.shape)

ResNet-18 output shape: torch.Size([32, 102])
ResNet-50 output shape: torch.Size([32, 102])


In [34]:
max_batches = 500

print("Training updates:", max_batches)

Training updates: 500


In [36]:
resnet18_parameters = sum(
    parameter.numel()
    for parameter in resnet18.parameters()
)

resnet50_parameters = sum(
    parameter.numel()
    for parameter in resnet50.parameters()
)

print("ResNet-18 total parameters:", resnet18_parameters)
print("ResNet-50 total parameters:", resnet50_parameters)

ResNet-18 total parameters: 11228838
ResNet-50 total parameters: 23717030
